# French Baby Names

First names registered in France by department, 1900-2020
([dpt2020.csv](https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv)).

1. Evolution over time - line chart with search
2. Regional spread - choropleth map
3. Gender split - centered butterfly chart

In [1]:
# !pip install -r requirements.txt

In [2]:
import json
import os
import time
from urllib.request import urlopen

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# Each chart is exported as a standalone HTML file in this folder
os.makedirs('exports', exist_ok=True)

## Data

Load the CSV, then keep valid years and real names.

In [3]:
url = "https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv"

# ~79 MB; the server connection sometimes drops mid-download, so fetch it once to a
# local cache (with a few retries) and read from there. Delete dpt2020.csv to refresh.
csv_file = "dpt2020.csv"
if not os.path.exists(csv_file):
    for attempt in range(5):
        try:
            with urlopen(url) as resp, open(csv_file, "wb") as f:
                f.write(resp.read())
            break
        except Exception:
            if os.path.exists(csv_file):
                os.remove(csv_file)
            if attempt == 4:
                raise
            time.sleep(3)

df = pd.read_csv(csv_file, sep=';', dtype={'annais': str, 'dpt': str})
df.head()

,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


In [4]:
df.columns = ['sex', 'name', 'year', 'dept', 'births']
df = df[df.name != '_PRENOMS_RARES']
df = df[df.year != 'XXXX']
df['year'] = df.year.astype(int)

## 1. Evolution over time

Evolution of French baby name popularity from 1900 to 2020. All top-300 names
start as faint gray lines. The user **first chooses how many names to compare**,
and exactly that many name dropdowns appear to fill in.

**Design**
- X-axis: Year (1900-2020)
- Y-axis: Total births per year (log scale, summed across departments and genders)
- Background: all top-300 names as thin gray lines (toggleable)
- Highlight: selected names drawn thick and colored, with end-of-line labels
- Interaction: a **"How many names?"** selector (1 ... `MAX_SLOTS`) drives a control
  panel that renders that many name dropdowns on the fly, each listing every name.

**Why a custom HTML page?** A compiled Vega-Lite chart always renders *all* of its
bound inputs -- a parameter cannot show or hide them. So this version exports the
chart with unbound `name1 ... nameN` parameters and wraps it in a small custom page
whose JavaScript builds the dropdowns dynamically and pushes the chosen names into
the chart via `view.signal(...)`.

In [ ]:
import unicodedata

# Normalize accents so spelling variants collapse into one name:
# LEO <- LÉO, MAEL <- MAËL, NOEMIE <- NOÉMIE, etc. INSEE stores accented and
# unaccented forms as separate records, which otherwise splits a single name
# into two short, incomplete lines. Stripping diacritics merges them.
def strip_accents(s):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )

evo = df[(df.year >= 1900) & (df.year <= 2020)].copy()
evo['name'] = evo['name'].map(strip_accents)

# Aggregate: total births per (name, year) -- sums across all departments,
# genders, AND accent variants of the same name.
yearly = (
    evo.groupby(['name', 'year'], as_index=False)['births'].sum()
    .rename(columns={'births': 'count'})
)

print(f"Unique names : {yearly['name'].nunique():,}")
print(f"Year range   : {yearly['year'].min()} - {yearly['year'].max()}")
print(f"Total rows   : {len(yearly):,}")

In [ ]:
# Keep only the top-N names by cumulative births to keep the chart responsive.
TOP_N = 300

top_names = (
    yearly.groupby('name')['count']
    .sum()
    .nlargest(TOP_N)
    .index.tolist()
)

# Exclude count <= 3: INSEE records rare occurrences as exactly 3 (privacy floor),
# which causes hundreds of names to pile up on the same Y position and form a
# visible horizontal strip at the bottom of the spaghetti chart.
df_plot = yearly[(yearly['name'].isin(top_names)) & (yearly['count'] > 3)].copy()
print(f"Plotting {TOP_N} names x {yearly['year'].nunique()} years = {len(df_plot):,} rows")

In [ ]:
# Lift Altair's default 5 000-row cap
alt.data_transformers.disable_max_rows()

# ── Two separate sources ──────────────────────────────────────────────────────
# Both layers use the same count > 3 floor so no data point falls below domainMin=4.
yearly_fg = yearly[yearly['count'] > 3]
base_bg = alt.Chart(df_plot)    # top-300 gray backdrop  (already filtered count > 3)
base_fg = alt.Chart(yearly_fg)  # all names, highlighted (filtered count > 3)

# ── Parameters WITHOUT bindings ──────────────────────────────────────────────
# name1 … nameMAX_SLOTS exist as plain signals (no native <select>). The custom
# HTML page (next save cell) drives them via view.signal() from dropdowns it
# builds dynamically. MAX_SLOTS is the upper bound the user can dial up to.
all_names = sorted(yearly_fg['name'].unique().tolist())

MAX_SLOTS   = 10
DEFAULT_NUM = 3                       # dropdowns shown on first load
defaults    = ['LEO', 'ALICE', 'MAEL']  # pre-filled names for the first slots

slot_params = [
    alt.param(f'name{i}', value=(defaults[i - 1] if i - 1 < len(defaults) else ''))
    for i in range(1, MAX_SLOTS + 1)
]

# Toggle for the gray background spaghetti. The custom HTML page exposes this as
# a checkbox and drives it via view.signal('show_bg', ...). The bg layer is kept
# or dropped with transform_filter('show_bg'): when the signal is false the
# layer renders nothing, masking all the dark-gray backdrop curves at once.
show_bg = alt.param('show_bg', value=True)

# Highlight a line if its name matches ANY slot. Empty slots never match.
param_array = '[' + ', '.join(f'name{i}' for i in range(1, MAX_SLOTS + 1)) + ']'
SELECT_FILTER = f'indexof({param_array}, datum.name) >= 0'

x_enc = alt.X('year:Q', title='Year', axis=alt.Axis(format='d', tickCount=13))

# domainMin=4 matches the count > 3 filter applied to both layers, so the axis
# floor aligns with the lowest data point and no line can overflow below the X-axis.
y_enc = alt.Y(
    'count:Q',
    title='Total births (log scale)',
    scale=alt.Scale(type='log', zero=False, domainMin=4, nice=False),
    axis=alt.Axis(format='~s')
)

TOOLTIP = [
    alt.Tooltip('name:N',  title='Name'),
    alt.Tooltip('year:Q',  title='Year',         format='d'),
    alt.Tooltip('count:Q', title='Total births', format=',')
]

# ── Layer 1 · ALL top-300 names — faint gray spaghetti ───────────────────────
# transform_filter('show_bg') lets the checkbox in the HTML page hide every
# backdrop curve at once when the signal is toggled off.
bg = base_bg.mark_line(
    strokeWidth=0.8,
    opacity=0.15,
    color='#888888'
).encode(
    x=x_enc,
    y=y_enc,
    detail='name:N',
    tooltip=TOOLTIP
).transform_filter('show_bg')

# ── Layer 2 · SELECTED names — colored, thick lines ──────────────────────────
fg = (
    base_fg.mark_line(strokeWidth=2.8, opacity=0.9)
    .encode(
        x=x_enc,
        y=y_enc,
        color=alt.Color(
            'name:N',
            scale=alt.Scale(scheme='tableau20'),
            legend=alt.Legend(title='Selected names', orient='top-right')
        ),
        detail='name:N',
        tooltip=TOOLTIP
    )
    .transform_filter(SELECT_FILTER)
)

# ── Layer 3 · End-of-line labels ─────────────────────────────────────────────
lbl = (
    base_fg.mark_text(align='left', dx=6, dy=-3, fontSize=11, fontWeight='bold')
    .encode(
        x=x_enc,
        y=y_enc,
        color=alt.Color('name:N', scale=alt.Scale(scheme='tableau20'), legend=None),
        text='name:N'
    )
    .transform_filter(SELECT_FILTER)
    .transform_joinaggregate(max_year='max(year)', groupby=['name'])
    .transform_filter('datum.year == datum.max_year')
)

# ── Compose ───────────────────────────────────────────────────────────────────
chart = (
    (bg + fg + lbl)
    .add_params(*slot_params, show_bg)
    .properties(
        title=alt.TitleParams(
            text='French Baby Names — Evolution Over Time (1900–2020)',
            subtitle='Choose how many names to compare, then pick each from its dropdown — Y axis is logarithmic',
            anchor='start',
            fontSize=16,
            fontWeight='bold',
            subtitleFontSize=12,
            subtitleColor='#666666'
        ),
        width=760,
        height=450,
        padding={'left': 10, 'right': 90, 'top': 10, 'bottom': 10}
    )
    .configure_view(strokeWidth=0)
    .configure_axis(grid=True, gridOpacity=0.2, gridColor='#e0e0e0')
)

chart

### Custom HTML page with dynamic dropdowns

The cell below takes the compiled chart spec and embeds it in a small standalone
page. A **"How many names?"** selector controls how many name dropdowns are
rendered; each change pushes the chosen names into the chart through
`view.signal('name{i}', value)`.

In [ ]:
import json

spec = chart.to_dict()

# Custom standalone page: control panel (number selector + dynamic name
# dropdowns) wrapping the embedded Vega-Lite view. Placeholders are filled in
# with .replace() to avoid clashing with the braces in the CSS/JS.
TEMPLATE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<title>French Baby Names — Choose How Many to Compare</title>
<script src="https://cdn.jsdelivr.net/npm/vega@6"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-lite@6.4.1"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@7"></script>
<style>
  body { font-family: -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif; margin: 24px; color:#222; }
  #panel { background:#f7f7f9; border:1px solid #e2e2e8; border-radius:8px; padding:14px 16px; margin-bottom:16px; max-width:900px; }
  #panel .row { display:flex; align-items:center; gap:10px; margin-bottom:10px; }
  #panel label { font-weight:600; font-size:14px; }
  #name-controls { display:flex; flex-wrap:wrap; gap:10px 18px; }
  .ctrl { display:flex; align-items:center; gap:6px; }
  .ctrl label { font-weight:500; }
  select { font-size:14px; padding:3px 6px; }
  .hint { color:#666; font-size:12px; }
  .bg-toggle { display:flex; align-items:center; gap:6px; cursor:pointer; }
  .bg-toggle input { width:16px; height:16px; cursor:pointer; }
</style>
</head>
<body>
  <div id="panel">
    <div class="row">
      <label for="num-select">How many names to compare?</label>
      <select id="num-select"></select>
      <span class="hint">Tip: click a dropdown and start typing to jump to a name.</span>
    </div>
    <div class="row">
      <label class="bg-toggle" for="bg-toggle">
        <input type="checkbox" id="bg-toggle" checked/>
        Show background curves
      </label>
      <span class="hint">Uncheck to hide the faint gray spaghetti backdrop.</span>
    </div>
    <div id="name-controls"></div>
  </div>
  <div id="vis"></div>

<script type="text/javascript">
const SPEC = __SPEC__;
const NAMES = __NAMES__;
const MAX = __MAX__;
const DEFAULT_NUM = __DEFAULT_NUM__;
const DEFAULTS = __DEFAULTS__;

let selected = Array.from({length: MAX}, (_, i) => DEFAULTS[i] || '');
let view = null;

const optionsHTML = '<option value="">— none —</option>' +
  NAMES.map(n => '<option value="' + n + '">' + n + '</option>').join('');

function applyAllSignals(k) {
  for (let i = 1; i <= MAX; i++) {
    view.signal('name' + i, i <= k ? (selected[i-1] || '') : '');
  }
  view.runAsync();
}

function renderDropdowns(k) {
  const c = document.getElementById('name-controls');
  c.innerHTML = '';
  for (let i = 1; i <= k; i++) {
    const wrap = document.createElement('div');
    wrap.className = 'ctrl';
    const lab = document.createElement('label');
    lab.textContent = 'Name ' + i + ':';
    const sel = document.createElement('select');
    sel.innerHTML = optionsHTML;
    sel.value = selected[i-1] || '';
    sel.addEventListener('change', (e) => {
      selected[i-1] = e.target.value;
      view.signal('name' + i, e.target.value);
      view.runAsync();
    });
    wrap.appendChild(lab);
    wrap.appendChild(sel);
    c.appendChild(wrap);
  }
  applyAllSignals(k);
}

vegaEmbed('#vis', SPEC, {actions: false}).then((res) => {
  view = res.view;
  const numSel = document.getElementById('num-select');
  for (let n = 1; n <= MAX; n++) {
    const o = document.createElement('option');
    o.value = n; o.textContent = n;
    if (n === DEFAULT_NUM) o.selected = true;
    numSel.appendChild(o);
  }
  numSel.addEventListener('change', (e) => renderDropdowns(parseInt(e.target.value, 10)));

  // Show/hide the gray background spaghetti by driving the show_bg signal.
  const bgToggle = document.getElementById('bg-toggle');
  bgToggle.addEventListener('change', (e) => {
    view.signal('show_bg', e.target.checked);
    view.runAsync();
  });

  renderDropdowns(DEFAULT_NUM);
}).catch(console.error);
</script>
</body>
</html>'''

html = (TEMPLATE
        .replace('__SPEC__', json.dumps(spec))
        .replace('__NAMES__', json.dumps(all_names))
        .replace('__MAX__', str(MAX_SLOTS))
        .replace('__DEFAULT_NUM__', str(DEFAULT_NUM))
        .replace('__DEFAULTS__', json.dumps(defaults)))

with open('exports/Viz1.html', 'w') as f:
    f.write(html)
print('Saved to exports/Viz1.html')

## 2. Regional spread

- Pick a name and a year. Each department is coloured by the **number of babies given that name there
that year** -- the darker, the more.
- The scale is fixed at **0 to 1000+** (1000 or more shows the
darkest shade), the same for every name and year. Exact count in the tooltip.

### Viz 2:  changes made, per peer review from the forum

| Reviewer | Comment on the forum | Change made |
|---|---|---|
| Ambroise & Yassine | the colour scale moves across years | colour scale **fixed at 0 to 1000+**, identical for every name and year |
| Andre & Anne | clunky to compare two names | added a **side-by-side** comparison of two names, on a **shared year** |
| Anne | sort the name dropdown alphabetically | dropdown **sorted alphabetically** |
| Nathan | use percentages instead of raw values | we **kept raw counts** (most direct reading) — *not adopted* |

In [7]:
# --- Viz 2 data: raw births per (name, dept, year) for a curated set of names ---
geo_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"

# small file, but use the same robust fetch-to-cache pattern as the CSV
geo_file = "france.json"
if not os.path.exists(geo_file):
    for attempt in range(5):
        try:
            with urlopen(geo_url) as resp, open(geo_file, "wb") as f:
                f.write(resp.read())
            break
        except Exception:
            if os.path.exists(geo_file):
                os.remove(geo_file)
            if attempt == 4:
                raise
            time.sleep(2)
with open(geo_file, encoding='utf-8') as r:
    france = json.load(r)

by_dept = df.groupby(['name', 'dept', 'year'], as_index=False).births.sum()

# Corsica is '20' in the data but 2A / 2B on the map
corse = by_dept[by_dept.dept == '20']
for code in '2A', '2B':
    by_dept = pd.concat([by_dept, corse.assign(dept=code)])

# curated names (Breton, Basque, national, trend-driven), sorted for the dropdown
picks = sorted(['KEVIN', 'ERWAN', 'MAEL', 'RONAN', 'AITOR', 'MAITE', 'MARIE', 'JEAN', 'MOHAMED'])
by_dept = by_dept[by_dept.name.isin(picks)][['name', 'dept', 'year', 'births']]

In [8]:
# --- Viz 2 maps: colour = raw births of a name in a department, fixed 0-1000+ scale ---
geo = alt.InlineData(values=france['features'])
lookup = alt.LookupData(geo, 'properties.code', ['type', 'geometry', 'properties'])
bg = alt.Chart(geo).mark_geoshape(fill='lightgray', stroke='white')

def carte(name_param, year_param, size, title):
    """One choropleth, driven by the params named `name_param` and `year_param`."""
    return (bg + alt.Chart(by_dept)
            .transform_filter(f'datum.name == {name_param} && datum.year == {year_param}')
            .transform_lookup('dept', from_=lookup).mark_geoshape(stroke='white')
            .encode(
                color=alt.Color('births:Q',
                                scale=alt.Scale(scheme='blues', domain=[0, 1000], clamp=True),
                                legend=alt.Legend(title='Babies named this',
                                                  labelExpr="datum.value >= 1000 ? '1000+' : datum.label")),
                tooltip=[alt.Tooltip('properties.nom:N', title='Department'),
                         alt.Tooltip('births:Q', title='Babies named this', format=',')])
            ).project('mercator').properties(width=size, height=size, title=title)

# one map: pick a name + year
name_sel = alt.param(name='sel_name', value='KEVIN', bind=alt.binding_select(options=picks, name='Name: '))
year_sel = alt.param(name='sel_year', value=1991, bind=alt.binding_range(min=1900, max=2020, step=1, name='Year: '))
(carte('sel_name', 'sel_year', 520, 'Regional spread')
 .add_params(name_sel, year_sel).save('exports/Viz2.html', inline=True))

# side-by-side: two names, one shared year
nameL = alt.param(name='nameL', value='KEVIN', bind=alt.binding_select(options=picks, name='Left name: '))
nameR = alt.param(name='nameR', value='JEAN', bind=alt.binding_select(options=picks, name='Right name: '))
cmp_year = alt.param(name='cmp_year', value=1991, bind=alt.binding_range(min=1900, max=2020, step=1, name='Year (shared): '))
((carte('nameL', 'cmp_year', 330, 'Left') | carte('nameR', 'cmp_year', 330, 'Right'))
 .resolve_scale(color='shared').add_params(nameL, nameR, cmp_year)
 .save('exports/Viz2_comparison.html', inline=True))

print('Exported exports/Viz2.html and exports/Viz2_comparison.html')

Exported exports/Viz2.html and exports/Viz2_comparison.html


## 3. Gender split


### Viz 3: Centered Butterfly Chart (Improvements)

Based on the peer feedback received, the following improvements were made to this visualization:

* **Enforced Symmetrical Scaling:** Addressed feedback regarding distorted axes by using an "invisible bounds" trick (`transform_joinaggregate` and transparent marks). This calculates the maximum absolute births for the selected name and forces the X-axis to scale symmetrically, ensuring the zero-line stays perfectly centered.
* **Dynamic Unisex Pre-selection Filter:** Replaced the static, hardcoded list of names with a dynamic Pandas calculation. The dropdown now automatically filters for and displays only truly unisex names (where the minority gender accounts for >10% of total births, with a baseline of >20,000 total births).
* **Enhanced Tooltips with Percentage Share:** Clarified the evolution of the gender split by adding calculations to the tooltip that compute and display the exact percentage share (e.g., "87.0%") alongside the absolute volume of births.
* **Improved Temporal Granularity:** Refined the Y-axis grouping from 20-year blocks to standard 10-year decades to provide better visual granularity for tracking rapid historical shifts.

In [9]:
gender = df.groupby(['name', 'sex', 'year'], as_index=False).births.sum()
gender['decade'] = (gender.year // 10) * 10
gender = gender.groupby(['name', 'sex', 'decade'], as_index=False).births.sum()

# --- Dynamic unisex name detection ---
# Total births per name per sex, pivoted so each sex is a column
sex_totals = (
    df.groupby(['name', 'sex']).births.sum()
    .unstack(fill_value=0)
    .rename(columns={1: 'male', 2: 'female'})
)
sex_totals['total'] = sex_totals['male'] + sex_totals['female']
sex_totals['minority_pct'] = sex_totals[['male', 'female']].min(axis=1) / sex_totals['total']

bfly_names = (
    sex_totals[
        (sex_totals['minority_pct'] > 0.10) &
        (sex_totals['total'] > 20_000)
    ]
    .index.sort_values()
    .tolist()
)

gender = gender[gender.name.isin(bfly_names)]

name_box = alt.binding_select(options=bfly_names, name='Name: ')
name_pick = alt.param(name='bf_name', value=bfly_names[0], bind=name_box)

base = (
    alt.Chart(gender)
    .transform_filter('datum.name == bf_name')
    .transform_joinaggregate(max_b='max(births)')
    .transform_joinaggregate(decade_total='sum(births)', groupby=['decade'])
    .transform_calculate(
        val='datum.sex == 1 ? -datum.births : datum.births',
        label='datum.sex == 1 ? "Male" : "Female"',
        sym_max='datum.max_b',
        sym_min='-datum.max_b',
        pct='datum.births / datum.decade_total'
    )
)

bars = base.mark_bar().encode(
    y=alt.Y('decade:O', sort='descending', title=None),
    x=alt.X('val:Q', title='Births', axis=alt.Axis(labelExpr='abs(datum.value)')),
    color=alt.Color('label:N',
        scale=alt.Scale(domain=['Male', 'Female'], range=['steelblue', 'darkorange']),
        legend=alt.Legend(orient='top', title=None)),
    tooltip=[alt.Tooltip('label:N', title='Sex'),
             alt.Tooltip('decade:O', title='Decade'),
             alt.Tooltip('births:Q', title='Births'),
             alt.Tooltip('pct:Q', title='Share', format='.1%')]
)

dummy_min = base.mark_point(opacity=0).encode(
    y=alt.Y('decade:O', sort='descending'),
    x=alt.X('sym_min:Q')
)

dummy_max = base.mark_point(opacity=0).encode(
    y=alt.Y('decade:O', sort='descending'),
    x=alt.X('sym_max:Q')
)

viz3 = (
    alt.layer(bars, dummy_min, dummy_max)
    .add_params(name_pick)
    .properties(width=500, height=300, title='Gender split over time')
)
viz3.save('exports/Viz3.html', inline=True)
print('Exported to exports/Viz3.html')



Exported to exports/Viz3.html
